# Class 4b — LLM Fundamentals (UCSC Extension)
# Homework 4b: LangChain Chains + ReAct Agents

**Jason Lim**

## Assignment
**b-1)** In the LangChain class example, the notebook implemented `llm_chain = title | character | story`.  
Add one more chaining block called **theme**. The new chain:  
`llm_chain = title | character | story | theme`  
Create a **new** story summary prompt (not the class/medical-seminar example).

**4b-2)** Write a **new** ReAct prompt that uses both **search** and **calculator** capabilities  
(different from the MacBook Mini / CAD class example).

## Theme continuity
Story summary stays in the **robotics / embodied AI** world from Homework 4a.

## References
- Class lab 4b / Hands-On LLM Ch. 7 (Advanced Text Generation — chains & agents)
- Colab lab: https://colab.research.google.com/drive/1IUv1fiFb4tGWyOgOGJEO5TcBNC0V7IFq


## Setup

Run in **Google Colab** with a **GPU** (Runtime → Change runtime type → T4).

For the ReAct section, add your OpenAI key as a Colab secret named `OPENAI_API_KEY`  
(or paste it into the cell below — do not commit real keys).


In [ ]:
%%capture
!pip install -q "langchain>=0.1.17" "openai>=1.13.3" "langchain_openai>=0.1.6" \
  "transformers>=4.40.1" "datasets>=2.18.0" "accelerate>=0.27.2" \
  "sentence-transformers>=2.5.1" "duckduckgo-search>=5.2.2" "langchain_community"
!CMAKE_ARGS="-DLLAMA_CUDA=on" pip install -q llama-cpp-python==0.2.69


In [ ]:
# Download Phi-3 mini (GGUF) for local LangChain chains
!wget -q -nc https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf


In [ ]:
from langchain import LlamaCpp

llm = LlamaCpp(
    model_path="Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False,
)


---
## b-1) Multiple chains — add a `theme` block

Class lab chain: `title | character | story`  
**Homework chain:** `title | character | story | theme`

Each step feeds the next:
1. **title** — from the story summary
2. **character** — from summary + title
3. **story** — from summary + title + character
4. **theme** — one-sentence moral / thematic takeaway from the full story


In [ ]:
from langchain import PromptTemplate, LLMChain

# Chain 1: title
template = """<s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>"""
title_prompt = PromptTemplate(template=template, input_variables=["summary"])
title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

# Chain 2: character
template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}. Use only two sentences.<|end|>
<|assistant|>"""
character_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title"]
)
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

# Chain 3: story
template = """<s><|user|>
Create a story about {summary} with the title {title}. The main character is: {character}. Only return the story and it cannot be longer than one paragraph.<|end|>
<|assistant|>"""
story_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title", "character"]
)
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

# Chain 4 (NEW): theme — moral / thematic takeaway
template = """<s><|user|>
Based on this story, state the central theme in one clear sentence.
Summary: {summary}
Title: {title}
Main character: {character}
Story: {story}
Only return the theme sentence.<|end|>
<|assistant|>"""
theme_prompt = PromptTemplate(
    template=template,
    input_variables=["summary", "title", "character", "story"],
)
theme = LLMChain(llm=llm, prompt=theme_prompt, output_key="theme")

# Full homework chain
llm_chain = title | character | story | theme
print("Chain ready:", "title | character | story | theme")


### New story summary prompt (b-1)

| | Prompt |
|---|--------|
| Class / prior example | *a very difficult to understand medical seminar* |
| **My new prompt** | **a warehouse robot that secretly writes poetry about conveyor belts at night** |

Why this prompt: it still exercises the full pipeline (title → character → story → theme) but shifts from a dry seminar topic to a robotics vignette with a clear emotional/thematic payoff for the new `theme` block.


In [ ]:
MY_STORY_SUMMARY = (
    "a warehouse robot that secretly writes poetry about conveyor belts at night"
)

result = llm_chain.invoke(MY_STORY_SUMMARY)

print("=== summary ===")
print(result.get("summary", MY_STORY_SUMMARY))
print("\n=== title ===")
print(result["title"])
print("\n=== character ===")
print(result["character"])
print("\n=== story ===")
print(result["story"])
print("\n=== theme (new block) ===")
print(result["theme"])


---
## 4b-2) ReAct agent — new prompt using search + calculator

The class ReAct example used a MacBook price lookup + currency conversion.  
Below: same **duckduck** (search) + **Calculator** (llm-math) tools, **new question**.

| | Prompt |
|---|--------|
| Class / prior example | *What is the current price of a MacBook Mini in USD? How much would it cost in Canadian dollars?* |
| **My new prompt** | *What is the current starting price of a Unitree Go2 robot dog in USD? How much would three of them cost in Japanese yen if the exchange rate is 150 JPY for 1 USD?* |

This forces the agent to:
1. **Search** for a current robot product price
2. **Calculate** `price × 3 × 150` for the yen total


In [ ]:
import os
from langchain_openai import ChatOpenAI

# Prefer Colab secret; fall back to env var already set in the runtime
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError(
            "Set OPENAI_API_KEY as a Colab secret or environment variable before running ReAct."
        )

openai_llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)


In [ ]:
from langchain import PromptTemplate
from langchain.agents import load_tools, Tool, AgentExecutor, create_react_agent
from langchain.tools import DuckDuckGoSearchResults

react_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate(
    template=react_template,
    input_variables=["tools", "tool_names", "input", "agent_scratchpad"],
)

search = DuckDuckGoSearchResults()
search_tool = Tool(
    name="duckduck",
    description="A web search engine. Use this as a search engine for general queries.",
    func=search.run,
)

tools = load_tools(["llm-math"], llm=openai_llm)
tools.append(search_tool)

agent = create_react_agent(openai_llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
)


In [ ]:
MY_REACT_PROMPT = (
    "What is the current starting price of a Unitree Go2 robot dog in USD? "
    "How much would three of them cost in Japanese yen if the exchange rate "
    "is 150 JPY for 1 USD?"
)

agent_executor.invoke({"input": MY_REACT_PROMPT})


---
## Short reflection

| Part | What changed |
|------|----------------|
| **b-1 theme chain** | A fourth sequential LLM call turns the finished story into an explicit one-sentence theme, so the pipeline is not only generative but also interpretive. |
| **New story summary** | Robotics + night poetry gives the `theme` block something emotional to extract (creativity vs. industrial routine). |
| **4b-2 ReAct prompt** | Search finds a live Unitree Go2 price; calculator multiplies by quantity and FX rate — same tool pair as class, different domain. |
